# Build the Filing Index and Join It to Compustat

| Item | Description |
|---|---|
| Goal | Turn the 394 containers into one table with one row per 10-K filing: the CIK, the exact form type, the fiscal period, the SEC.gov URL the Extractor takes, and the GVKEY the filing belongs to. Only each filing's `metadata.json` is read out of the archive. This is where the sample is defined. |
| Input | `Raw/form10k/<year>/<year>-<month>.zip` from D3.1. Two WRDS files in `Raw/firm_id`, kept locally and never pushed: `crsp_compustat_link.dta`, the CRSP/Compustat Merged link table (GVKEY, CIK, LPERMNO, LINKTYPE, LINKPRIM, LINKDT, LINKENDDT, company name, ticker, SIC), and `sp500_constituents.dta`, the daily CRSP S&P 500 constituent list (PERMNO). |
| Output | In the D3 Processed folder: `filings.parquet`, one row per annual report with GVKEY, fiscal year, link-window flag, and the S&P 500 flag; `sample_firms.parquet`, one row per CIK with GVKEY and the S&P 500 flag. |
| Variables | • `cik_unpadded`: the CIK without leading zeros, the join key on both sides<br>• `gvkey`: Compustat firm identifier, kept as a zero-padded string<br>• `in_sp500`: the firm was ever in the S&P 500<br>• `fyear`: fiscal year by Compustat's rule, a period ending January to May belongs to the previous year<br>• `in_link_window`: the period end falls inside one of the firm's CRSP-Compustat link spells<br>• `period_imputed`: the period end was missing and set to the filing date minus 90 days<br>• `url_bare`: the shipped URL was a bare CIK directory and was rebuilt from the accession number |
| Run | Last run 2026-08-20; the notebook kept no cell outputs. Not rerun: the extraction and every later stage were built on this index. Step 11 reads the saved tables and was executed on 2026-10-31 against the copied files. |

## Step 1. Import packages

In [ ]:
import json
import re
import zipfile
from pathlib import Path

import pandas as pd

## Step 2. Define file locations

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D - Data Collection and Processing/D3 - SEC Form 10-K Corporate Filings")
NOTEBOOK_NAME = "D3.2 - SEC 10-K - Build Filing Index & Linkage"
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
CONTAINER_FOLDER = RAW_FOLDER / "form10k"  # written by D3.1
FIRM_ID_FOLDER = RAW_FOLDER / "firm_id"  # the two WRDS files

PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)

## Step 3. Set the form types, the link vintage, and the URL rule

Five form types are annual reports; the other three in the dataset are amendments. `10KSB` and `10KSB40` carry no hyphen in EDGAR, and writing them with one matches nothing. An open-ended link spell has no `LINKENDDT` and is closed at the link file's vintage instead of being dropped. A filing URL that ends at the CIK directory names a company, not a filing.

In [ ]:
ANNUAL_FORMS = ("10-K", "10-K405", "10KSB", "10KSB40", "10-KT")
VINTAGE_END = pd.Timestamp("2025-12-31")  # the link file's last vintage; closes open-ended LINKENDDT rows
BARE_CIK = re.compile(r"^https://www\.sec\.gov/Archives/edgar/data/\d+/?$")  # a URL with nothing after the CIK

print(f"containers: {CONTAINER_FOLDER}")
print(f"annual forms kept: {ANNUAL_FORMS}")

## Step 4. Build the sample frame from the link table

The usable-link convention keeps `LINKTYPE` LC or LU and `LINKPRIM` P or C. Taking P alone drops 4,446 firms, 17.6 percent of the sample. The filtered table maps 25,207 CIKs to 25,207 GVKEYs one to one, so a filing maps to a firm by CIK alone. The CIK is zero-padded in the link file and unpadded in the filing metadata, so the join key is the unpadded string.

In [ ]:
link = pd.read_stata(FIRM_ID_FOLDER / "crsp_compustat_link.dta", convert_categoricals=False)
before = len(link)

link["cik"] = link["cik"].astype(str).str.strip()
link = link.loc[link["cik"].ne("") & link["cik"].ne("nan")]
after_cik = len(link)

link = link.loc[link["LINKTYPE"].isin(["LC", "LU"]) & link["LINKPRIM"].isin(["P", "C"])]
after_link = len(link)

link["LINKDT"] = pd.to_datetime(link["LINKDT"], errors="coerce")
link["LINKENDDT"] = pd.to_datetime(link["LINKENDDT"], errors="coerce").fillna(VINTAGE_END)

print(f"link rows                       {before:>8,}")
print(f"  with a CIK                    {after_cik:>8,}  (dropped {before - after_cik:,})")
print(f"  usable link type and primacy  {after_link:>8,}  (dropped {after_cik - after_link:,})")
print(f"  distinct CIK                  {link['cik'].nunique():>8,}")
print(f"  distinct GVKEY                {link['GVKEY'].nunique():>8,}")

# The one-to-one mapping is asserted, not assumed: every downstream join is one-to-one because of it.
assert link["cik"].nunique() == link["GVKEY"].nunique(), "cik and GVKEY are not in bijection"

# A padded CIK looked up at sec-api returns total=0 with HTTP 200, indistinguishable from a firm that never filed.
assert link["cik"].dtype == object, f"cik is {link['cik'].dtype}, expected object"
assert link["cik"].str.fullmatch(r"\d{10}").all(), "not every cik is 10 digits"
link["cik_unpadded"] = link["cik"].str.lstrip("0")
assert link["cik_unpadded"].str.fullmatch(r"[1-9]\d*").all(), "unpadded cik is malformed"

# GVKEY is a zero-padded string too. Cast to int anywhere downstream it would lose the padding and break a merge.
assert link["GVKEY"].dtype == object, f"GVKEY is {link['GVKEY'].dtype}, expected object"
print(f"\ncik      dtype={link['cik'].dtype}  e.g. {link['cik'].iloc[0]!r} -> unpadded {link['cik_unpadded'].iloc[0]!r}")
print(f"GVKEY    dtype={link['GVKEY'].dtype}  e.g. {link['GVKEY'].iloc[0]!r} (width {link['GVKEY'].str.len().unique().tolist()})")

## Step 5. Flag S&P 500 membership

The constituent file is keyed on `PERMNO`, so membership reaches the sample through the link table's `LPERMNO`. The flag means the firm was ever in the index. It is used only to decide extraction order.

In [ ]:
sp = pd.read_stata(FIRM_ID_FOLDER / "sp500_constituents.dta", convert_categoricals=False, columns=["PERMNO"])
sp500_permnos = set(sp["PERMNO"].dropna().astype(int))

link["permno"] = pd.to_numeric(link["LPERMNO"], errors="coerce")
link["in_sp500"] = link["permno"].isin(sp500_permnos)

firms = link.groupby("cik", as_index=False).agg(
    gvkey=("GVKEY", "first"),
    cik_unpadded=("cik_unpadded", "first"),
    conm=("conm", "first"),
    tic=("tic", "first"),
    sic=("sic", "first"),
    permno=("permno", "first"),
    in_sp500=("in_sp500", "any"),
    link_start=("LINKDT", "min"),
    link_end=("LINKENDDT", "max"),
)
print(f"sample firms      {len(firms):>7,}")
print(f"  ever in S&P 500 {int(firms['in_sp500'].sum()):>7,}")
print(f"  matched PERMNOs {firms['permno'].isin(sp500_permnos).sum():>7,} of {len(sp500_permnos):,} index PERMNOs")

## Step 6. Read every filing's metadata out of the containers

Only `metadata.json` is read. Unzipping 34 GB of documents to build a table of a few hundred thousand rows is not needed. The Extractor fetches from SEC.gov by URL anyway.

In [ ]:
records = []
containers = sorted(CONTAINER_FOLDER.rglob("*.zip"))
print(f"{len(containers)} containers on disk")

for n, path in enumerate(containers, 1):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not name.endswith("metadata.json"):
                continue
            meta = json.loads(z.read(name))
            records.append({
                "accessionNo": meta.get("accessionNo"),
                "cik_unpadded": str(meta.get("cik", "")).strip().lstrip("0"),  # a string in some containers, a number in others
                "formType": meta.get("formType"),
                "filedAt": meta.get("filedAt"),
                "periodOfReport": meta.get("periodOfReport"),
                "companyName": meta.get("companyName"),
                "ticker": meta.get("ticker") or None,
                "filingUrl": meta.get("linkToFilingDetails"),
                "container": path.relative_to(CONTAINER_FOLDER).as_posix(),
            })
    if n % 100 == 0:
        print(f"  {n}/{len(containers)} containers, {len(records):,} filings")

filings = pd.DataFrame.from_records(records)
print(f"\n{len(filings):,} filings read")

bad = filings.loc[~filings["cik_unpadded"].str.fullmatch(r"[1-9]\d*", na=False)]
print(f"filings whose cik is not a clean unpadded number: {len(bad):,}")
if len(bad):
    print(bad[["accessionNo", "cik_unpadded"]].head().to_string(index=False))
print(f"cik_unpadded dtype={filings['cik_unpadded'].dtype}  e.g. {filings['cik_unpadded'].iloc[0]!r}")
print()
print(filings["formType"].value_counts().to_string())

## Step 7. Keep the annual reports and join to the sample

Two restrictions, each printed. The match rate of the join key is printed too, because a key-shape mismatch produces an empty merge, not an error.

In [ ]:
before_rows = len(filings)
filings = filings.loc[filings["formType"].isin(ANNUAL_FORMS)]
after_forms = len(filings)

key_overlap = len(set(filings["cik_unpadded"]) & set(firms["cik_unpadded"]))
print(f"  distinct CIKs: {filings['cik_unpadded'].nunique():,} in the archive, {len(firms):,} in the sample, {key_overlap:,} shared")

filings = filings.merge(firms[["cik_unpadded", "cik", "gvkey", "conm", "in_sp500"]], on="cik_unpadded", how="inner", validate="many_to_one")
after_sample = len(filings)

print(f"filings in the archive      {before_rows:>8,}")
print(f"  annual report form types  {after_forms:>8,}  (dropped {before_rows - after_forms:,})")
print(f"  in the Compustat sample   {after_sample:>8,}  (dropped {after_forms - after_sample:,})")
print(f"  distinct firms            {filings['gvkey'].nunique():>8,}")
print(f"  S&P 500 filings           {int(filings['in_sp500'].sum()):>8,}")

## Step 8. Assign the fiscal year and the link-window flag

A missing period of report is set to the filing date minus 90 days and flagged. The link window decides panel membership, not identity. A filing whose link spell has closed is still that firm's filing. Membership is tested against each spell, because collapsing spells to a min and max fills the gaps and overstates coverage.

In [ ]:
filings["filedAt"] = pd.to_datetime(filings["filedAt"], format="mixed", utc=True)
period = pd.to_datetime(filings["periodOfReport"], errors="coerce", utc=True)

filings["period_imputed"] = period.isna()
period = period.fillna(filings["filedAt"] - pd.Timedelta(days=90))
filings["periodEnd"] = period.dt.tz_localize(None)

# Compustat convention: a period ending January to May belongs to the prior fiscal year.
filings["fyear"] = filings["periodEnd"].dt.year.where(filings["periodEnd"].dt.month > 5, filings["periodEnd"].dt.year - 1)

spells = link[["cik", "LINKDT", "LINKENDDT"]]
paired = filings[["accessionNo", "cik", "periodEnd"]].merge(spells, on="cik", how="left")
inside = (paired["periodEnd"] >= paired["LINKDT"]) & (paired["periodEnd"] <= paired["LINKENDDT"])
in_window = inside.groupby(paired["accessionNo"]).any()
filings["in_link_window"] = filings["accessionNo"].map(in_window).fillna(False)

print(f"period imputed from filedAt : {int(filings['period_imputed'].sum()):,} ({filings['period_imputed'].mean():.2%})")
print(f"inside the CRSP-Compustat link window : {int(filings['in_link_window'].sum()):,} ({filings['in_link_window'].mean():.1%})")
print()
print(filings.groupby(filings["fyear"] // 5 * 5).size().rename("filings").to_string())

## Step 9. Repair the filing URL

For 25,675 filings, 16.2 percent, the shipped URL is a bare CIK directory, which names a company and not a filing. All of them were filed on or before 1999-05-21, but the rule is keyed on the URL's shape, not on age. The URL is rebuilt from the accession number. Verified on twelve affected filings: all twelve failed on the shipped URL and all twelve returned real Item 1 text on the rebuilt one.

In [ ]:
filings["url_bare"] = filings["filingUrl"].fillna("").str.match(BARE_CIK)


def accession_url(cik, accession):
    """The EDGAR accession directory, which the Extractor can resolve."""
    return f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession.replace('-', '')}/"


filings["filingUrl_original"] = filings["filingUrl"]
filings.loc[filings["url_bare"], "filingUrl"] = [
    accession_url(c, a) for c, a in zip(filings.loc[filings["url_bare"], "cik_unpadded"], filings.loc[filings["url_bare"], "accessionNo"])
]

n_bare = int(filings["url_bare"].sum())
print(f"filing URLs that were a bare CIK directory : {n_bare:,} ({n_bare / len(filings):.1%})")
if n_bare:
    filed = pd.to_datetime(filings.loc[filings["url_bare"], "filedAt"], utc=True, format="mixed")
    print(f"  latest filedAt among them               : {filed.max().date()}")
    print(f"  of these, fiscal year >= 2000           : {int(filings.loc[filings['url_bare'], 'fyear'].ge(2000).sum()):,}")
    example = filings.loc[filings["url_bare"]].iloc[0]
    print("  example repair:")
    print(f"    was {example['filingUrl_original']}")
    print(f"    now {example['filingUrl']}")

left = int(filings["filingUrl"].fillna("").str.match(BARE_CIK).sum())
assert left == 0, f"{left} filing URLs are still a bare CIK directory"
print("\nno bare-CIK URLs remain in the index")

## Step 10. Save the index

In [ ]:
filings = filings.sort_values(["cik_unpadded", "periodEnd"]).reset_index(drop=True)
filings.to_parquet(PROCESSED_FOLDER / "filings.parquet", index=False)
firms.to_parquet(PROCESSED_FOLDER / "sample_firms.parquet", index=False)

print(f"filings.parquet      {len(filings):>8,} rows")
print(f"sample_firms.parquet {len(firms):>8,} rows")
print()
sp500 = filings.loc[filings["in_sp500"]]
print(f"S&P 500 subset: {len(sp500):,} filings across {sp500['gvkey'].nunique():,} firms -> {len(sp500) * 6:,} extractor calls")
print(f"full sample   : {len(filings):,} filings -> {len(filings) * 6:,} extractor calls")

## Step 11. Check the saved index

This cell reads the two saved tables, so it can run at any time. Five checks stop the notebook if they fail. The sixth is the independent one: SEC's `companyName` and Compustat's `conm` come from different vendors. They agree after normalization only if the CIK join is right.

- `accessionNo` is unique.
- Every row carries a resolvable filing URL.
- No `cik_unpadded` keeps a leading zero.
- Every filing maps to exactly one GVKEY.
- The form types are exactly the five annual ones.
- Company names agree on more than 80 percent of mapped firms, and ten random rows are printed to read.

In [1]:
filings = pd.read_parquet(PROCESSED_FOLDER / "filings.parquet")
firms = pd.read_parquet(PROCESSED_FOLDER / "sample_firms.parquet")
print(f"filings.parquet {len(filings):,} rows, sample_firms.parquet {len(firms):,} rows\n")

dupes = int(filings["accessionNo"].duplicated().sum())
assert dupes == 0, f"{dupes} duplicate accessionNo: a filing was read twice"
print(f"1) {len(filings):,} filings, all accessionNo distinct")

no_url = int(filings["filingUrl"].isna().sum())
assert no_url == 0, f"{no_url} filings have no filing URL"
bare_left = int(filings["filingUrl"].str.match(BARE_CIK).sum())
assert bare_left == 0, f"{bare_left} filings still carry a bare-CIK URL"
print(f"2) every filing carries a resolvable extractor URL ({int(filings['url_bare'].sum()):,} of them repaired)")

padded = int(filings["cik_unpadded"].str.startswith("0").sum())
assert padded == 0, f"{padded} CIKs still carry a leading zero"
print("3) no padded CIKs")

per_filing = filings.groupby("accessionNo")["gvkey"].nunique()
assert (per_filing == 1).all(), f"{int((per_filing > 1).sum())} filings map to >1 GVKEY"
print("4) every filing maps to exactly one GVKEY")

extra = set(filings["formType"]) - set(ANNUAL_FORMS)
assert not extra, f"unexpected form types survived the filter: {extra}"
print(f"5) form types are exactly {sorted(set(filings['formType']))}")

SEED = 20260820
SUFFIXES = re.compile(r"(inc|incorporated|corp|corporation|co|company|ltd|limited|llc|lp|plc|holdings?|group|the|new|cl [ab]|-cl [ab]|adr|sa|nv|ag)")


def normalise(name):
    """Case, punctuation, and corporate suffixes removed; what is left is the name."""
    s = re.sub(r"[^a-z0-9 ]", " ", str(name).lower())
    s = SUFFIXES.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()


def agrees(sec_name, cs_name):
    """Generous on purpose: a prefix match or half the tokens shared counts as agreement."""
    if not sec_name or not cs_name:
        return False
    if sec_name == cs_name or sec_name.startswith(cs_name) or cs_name.startswith(sec_name):
        return True
    a, b = set(sec_name.split()), set(cs_name.split())
    return bool(a & b) and len(a & b) / min(len(a), len(b)) >= 0.5


# Each firm's latest filing: conm is the name Compustat holds today, so an old header would measure renaming, not the join.
check = filings.drop_duplicates("cik_unpadded", keep="last").copy()
check["name_agrees"] = [agrees(normalise(s), normalise(c)) for s, c in zip(check["companyName"], check["conm"])]
rate = check["name_agrees"].mean()
oldest = filings.drop_duplicates("cik_unpadded", keep="first")
rate_oldest = pd.Series([agrees(normalise(s), normalise(c)) for s, c in zip(oldest["companyName"], oldest["conm"])]).mean()

print(f"6) company names agree on {int(check['name_agrees'].sum()):,} of {len(check):,} mapped firms ({rate:.1%}), comparing each firm's latest filing header")
print(f"   same test on each firm's earliest filing header: {rate_oldest:.1%}; the gap is renaming over the sample period")
print("\n   ten random mapped filings, read these:")
for row in filings.sample(10, random_state=SEED).itertuples():
    ok = "OK " if agrees(normalise(row.companyName), normalise(row.conm)) else "?? "
    print(f"   [{ok}] cik {row.cik_unpadded:>8}  gvkey {row.gvkey}  fyear {row.fyear}  {row.formType}")
    print(f"         SEC filing header : {row.companyName}")
    print(f"         Compustat conm    : {row.conm}")
assert rate > 0.80, f"only {rate:.1%} of mapped firms have agreeing names; the CIK join is probably wrong"
print(f"\n   name agreement {rate:.1%}: the CIK-to-GVKEY join holds")

filings.parquet 158,396 rows, sample_firms.parquet 25,207 rows

1) 158,396 filings, all accessionNo distinct
2) every filing carries a resolvable extractor URL (25,675 of them repaired)
3) no padded CIKs
4) every filing maps to exactly one GVKEY
5) form types are exactly ['10-K', '10-K405', '10-KT']
6) company names agree on 14,966 of 15,508 mapped firms (96.5%), comparing each firm's latest filing header
   same test on each firm's earliest filing header: 79.3%; the gap is renaming over the sample period

   ten random mapped filings, read these:
   [OK ] cik   862510  gvkey 023006  fyear 2000  10-K405
         SEC filing header : TRANSATLANTIC HOLDINGS INC
         Compustat conm    : TRANSATLANTIC HOLDINGS INC
   [OK ] cik    55529  gvkey 006410  fyear 2015  10-K
         SEC filing header : KEWAUNEE SCIENTIFIC CORP /DE/
         Compustat conm    : KEWAUNEE SCIENTIFIC CORP
   [?? ] cik  1412347  gvkey 184569  fyear 2010  10-K
         SEC filing header : Oxford Resource Partners LP